In [1]:
import pandas as pd

In [2]:
df = pd.read_json('mydb.Enrich_logs.json', orient='records')


In [3]:
df.head()

,_id,user_id,item_id,action,category,device,location,timestamp
0,{'$oid': '6978d5cccb9dff3ea8c09d0e'},183.81.126.150,game-minecraft-ban-sung,view_news,News,K (Android),"Ho Chi Minh City, Vietnam",{'$date': '2026-01-04T16:55:18.674Z'}
1,{'$oid': '6978d5cccb9dff3ea8c09d0f'},171.246.103.28,dien-thoai,view_product,dien-thoai,Mac,"Ho Chi Minh City, Vietnam",{'$date': '2026-01-04T16:09:15.223Z'}
2,{'$oid': '6978d5cccb9dff3ea8c09d10'},72.14.201.135,huong-dan-dang-nhap-zalo-web-bang-ma-qr-va-so-...,view_news,News,Windows Device,Vietnam,{'$date': '2026-01-07T06:09:35.151Z'}
3,{'$oid': '6978d5cccb9dff3ea8c09d11'},123.27.186.174,dien-thoai,view_product,dien-thoai,K (Android),"Haiphong, Vietnam",{'$date': '2026-01-05T02:24:25.204Z'}
4,{'$oid': '6978d5cccb9dff3ea8c09d12'},171.232.110.64,dien-thoai,view_product,dien-thoai,iPhone,"Ho Chi Minh City, Vietnam",{'$date': '2026-01-06T17:45:11.147Z'}


In [4]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    format='mixed',
    errors='coerce',
    utc=True
)
df = df.drop(columns=["_id"])


In [5]:
df_rec = df[df['action'] == 'view_product']

In [19]:
import pandas as pd
import numpy as np
from collections import defaultdict
from itertools import combinations
from math import sqrt


class ItemItemCooccurrence:
    def __init__(self, session_gap_minutes=30, min_cooccurrence=1):
        self.session_gap = pd.Timedelta(minutes=session_gap_minutes)
        self.min_cooccurrence = min_cooccurrence

        self.cooccurrence = defaultdict(float)
        self.item_counts = defaultdict(int)
        self.similarity = defaultdict(dict)

    def _build_sessions(self, df):
        df = df.sort_values(["user_id", "timestamp"])

        sessions = []
        current_session = []
        last_user = None
        last_time = None

        for row in df.itertuples(index=False):
            if (
                row.user_id != last_user
                or last_time is None
                or row.timestamp - last_time > self.session_gap
            ):
                if current_session:
                    sessions.append(set(current_session))
                current_session = [row.item_id]
            else:
                current_session.append(row.item_id)

            last_user = row.user_id
            last_time = row.timestamp

        if current_session:
            sessions.append(set(current_session))

        return sessions

    def fit(self, df):
        sessions = self._build_sessions(df)

        for items in sessions:
            for item in items:
                self.item_counts[item] += 1

            for i, j in combinations(sorted(items), 2):
                self.cooccurrence[(i, j)] += 1
                self.cooccurrence[(j, i)] += 1

        self._compute_similarity()

    def _compute_similarity(self):
        for (i, j), cij in self.cooccurrence.items():
            if cij < self.min_cooccurrence:
                continue

            denom = sqrt(self.item_counts[i] * self.item_counts[j])
            if denom == 0:
                continue

            sim = cij / denom
            self.similarity[i][j] = sim

    def recommend(self, item_id, k=10):
        if item_id not in self.similarity:
            return "No recommendations available."

        return sorted(
            self.similarity[item_id].items(),
            key=lambda x: x[1],
            reverse=True
        )[:k]


In [21]:
df_rec['item_id'].value_counts()

item_id
dien-thoai                        34168
firework                          15687
tablet                             8464
iphone-16                          5593
thiet-bi-deo                       5294
iphone-17                          4787
laptop                             2794
iphone-15                          2276
iphone-14                          1164
tivi                                570
phu-kien                            364
ipad-air-m3                         219
ipad-a16                            163
apple-watch-series-11               121
ipad-air-m2                          88
redirect                             71
ipad-pro-m4                          67
apple-watch-ultra-3                  57
apple-watch-series-10                44
uu-dai-tra-gop                       42
apple-watch-se-3                     41
apple-watch-se-2                     29
may-doc-sach                         26
apple-watch-ultra-2                  26
category_iphone14               

In [22]:
model = ItemItemCooccurrence()
model.fit(df_rec)
res = model.recommend(item_id='iphone-16', k=10)


In [23]:
print(res)

[('dien-thoai', 0.1952904997022285), ('iphone-15', 0.19519356748009667), ('iphone-17', 0.16005446927777842), ('iphone-14', 0.12817342136646323), ('tablet', 0.05108767451260122), ('thiet-bi-deo', 0.04299964044745142), ('firework', 0.03666881515050074), ('laptop', 0.030971332110922477), ('redirect', 0.025140903411419786), ('phu-kien', 0.02239151172212139)]
